In [0]:
# Config cell — change model names here only, if a built-in model name changes
CATALOG = "hop"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
print(CATALOG, CHAT_MODEL, EMBED_MODEL)

In [0]:
# R1 — Auto Loader bronze ingestion of order events
# - Every line of every file is kept (duplicates included)
# - Adds _source_file, _ingest_ts; platform (and any later new field) arrives via schema evolution
from pyspark.sql import functions as F

LANDING = "/Volumes/hop/raw/landing/order_events/"
CHK = "/Volumes/hop/raw/landing/_checkpoints/bronze_order_events"

bronze_schema = """
  event_id STRING, order_id STRING, event_type STRING, event_ts STRING,
  customer_id STRING, restaurant_id STRING, zone_id STRING,
  delivery_class STRING, promised_minutes LONG, order_value DOUBLE,
  rider_id STRING, cancelled_by STRING, platform STRING
"""

events = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHK + "/schema")
    .option("cloudFiles.schemaHints", bronze_schema)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(LANDING)
    .withColumn("_source_file", F.col("_metadata.file_name"))
    .withColumn("_ingest_ts", F.current_timestamp())
)

q = (
    events.writeStream
    .option("checkpointLocation", CHK)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.bronze.order_events")
)
q.awaitTermination()
print("bronze rows:", spark.table(f"{CATALOG}.bronze.order_events").count())